# Reviewer benchmark: read-only results analysis

This notebook consumes the machine-readable outputs of `src/reviewer_benchmark.py`. It never tokenizes sequences, constructs features, or fits a model. Set `REVIEWER_BENCHMARK_OUTPUT` to the output directory (default: `results/reviewer_benchmark`) before running. Empty or partial output directories are expected while experiments are pending; cells report what is missing and continue.

The primary unit is an out-of-fold condition. Prefix distributions therefore include every prespecified prefix, while full-versus-filtered comparisons use the same fold-level metrics and resource metadata.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from fnmatch import fnmatch
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False

def _find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    return start

REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SRC = REPOSITORY_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
try:
    from utilities.benchmark_metrics import (
        chance_balanced_accuracy, fold_confidence_interval,
        paired_difference_confidence_interval, prediction_metrics,
    )
except ImportError:
    def chance_balanced_accuracy(n_classes: int) -> float:
        return 1.0 / int(n_classes)
    prediction_metrics = None
    fold_confidence_interval = None
    paired_difference_confidence_interval = None

def _repository_relative_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPOSITORY_ROOT / path

OUTPUT_ROOT = _repository_relative_path(os.environ.get("REVIEWER_BENCHMARK_OUTPUT", "results/reviewer_benchmark"))
split_manifest_value = os.environ.get("REVIEWER_BENCHMARK_SPLIT_MANIFEST")
SPLIT_MANIFEST_PATH = _repository_relative_path(split_manifest_value) if split_manifest_value else None
EXPORT_OUTPUTS = os.environ.get("REVIEWER_BENCHMARK_EXPORT", "0") == "1"
EXPORT_DIR = _repository_relative_path(os.environ.get("REVIEWER_BENCHMARK_EXPORT_DIR", str(OUTPUT_ROOT / "publication_exports")))
CONFIDENCE = 0.95
BOOTSTRAP_SEED = 20260901
print(f"Reading saved benchmark outputs from: {OUTPUT_ROOT.resolve()}")
print("Plotting available:", PLOTTING_AVAILABLE, "| Export enabled:", EXPORT_OUTPUTS)

## 1. Load and validate manifests, metrics, predictions, and resource tables

All tables are discovered recursively so both a consolidated run and one-directory-per-condition output are supported. JSON manifests that contain configuration dictionaries are retained separately from tabular manifests.

In [ ]:
TABLE_EXTENSIONS = {".csv", ".parquet", ".pq", ".json"}

def _read_table(path: Path) -> pd.DataFrame | None:
    try:
        if path.suffix.lower() in {".parquet", ".pq"}:
            return pd.read_parquet(path)
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)
        if path.suffix.lower() == ".json":
            value = json.loads(path.read_text())
            if isinstance(value, list):
                return pd.DataFrame(value)
            if isinstance(value, dict):
                # Failure records are scalar-key JSON objects; loading
                # them as one row keeps failed conditions visible in QC.
                # Nested resolved manifests are excluded by _discover's
                # filename patterns and are loaded separately below.
                return pd.DataFrame(value) if all(isinstance(item, list) for item in value.values()) else pd.DataFrame([value])
    except Exception as error:
        print(f"Could not read {path}: {type(error).__name__}: {error}")
    return None

def _discover(patterns: tuple[str, ...]) -> list[Path]:
    if not OUTPUT_ROOT.exists():
        return []
    excluded_roots = {EXPORT_DIR.resolve(), (OUTPUT_ROOT / "readable").resolve()}
    paths = [p for p in OUTPUT_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in TABLE_EXTENSIONS and not any(root == p.resolve().parent or root in p.resolve().parents for root in excluded_roots)]
    return sorted({p for p in paths if any(fnmatch(p.name.lower(), pattern) for pattern in patterns)})

def _concat_named(patterns: tuple[str, ...], name: str) -> pd.DataFrame:
    frames = []
    for path in _discover(patterns):
        frame = _read_table(path)
        if frame is not None and not frame.empty:
            frame = frame.copy()
            frame["_source_file"] = str(path)
            frames.append(frame)
    result = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    print(f"{name}: {len(result):,} rows from {len(frames)} file(s)")
    return result

metrics = _concat_named(("metrics.csv", "metrics.parquet", "metrics.pq"), "Metrics")
resources = _concat_named(("resources.csv", "resources.parquet", "resources.pq"), "Resources")
predictions = _concat_named(("predictions.csv", "predictions.parquet", "predictions.pq"), "Predictions")
conditions = _concat_named(("conditions*.csv", "conditions*.parquet", "conditions*.pq"), "Condition manifests")
split_manifest = _concat_named(("split_manifest*.csv", "split_manifest*.parquet", "split_manifest*.pq"), "Split manifests")
if SPLIT_MANIFEST_PATH and Path(SPLIT_MANIFEST_PATH).exists():
    extra_split = _read_table(Path(SPLIT_MANIFEST_PATH))
    if extra_split is not None:
        extra_split = extra_split.copy()
        extra_split["_source_file"] = str(Path(SPLIT_MANIFEST_PATH))
        split_manifest = pd.concat([split_manifest, extra_split], ignore_index=True, sort=False)
        print(f"Loaded split manifest from REVIEWER_BENCHMARK_SPLIT_MANIFEST: {len(extra_split):,} rows")
prefix_manifest = _concat_named(("prefix_manifest*.csv", "prefix_manifest*.parquet", "prefix_manifest*.pq"), "Prefix manifests")
failures = _concat_named(("*failure*.json", "*failure*.csv", "*failure*.parquet"), "Failure records")
resfinder = _concat_named(("*resfinder*.csv", "*resfinder*.parquet", "*resfinder*.pq"), "Optional ResFinder tables")
# Keep resolved run manifests available for provenance without treating their
# nested configuration dictionaries as tidy result tables.
resolved_manifests = []
for manifest_path in sorted(OUTPUT_ROOT.rglob("*manifest*.json")) if OUTPUT_ROOT.exists() else []:
    try:
        value = json.loads(manifest_path.read_text())
        if isinstance(value, dict):
            resolved_manifests.append({"path": str(manifest_path), **value})
    except Exception as error:
        print(f"Could not read manifest {manifest_path}: {error}")
print(f"Resolved JSON manifests: {len(resolved_manifests)}")

def _validate(frame: pd.DataFrame, required: set[str], name: str) -> bool:
    missing = sorted(required.difference(frame.columns))
    if frame.empty:
        print(f"{name}: empty (this is expected before experiments finish)")
        return False
    if missing:
        print(f"{name}: missing required columns {missing}; downstream summaries may be unavailable")
        return False
    print(f"{name}: schema OK ({len(frame.columns)} columns)")
    return True

metrics_ok = _validate(metrics, {"condition_id", "fold_id", "balanced_accuracy"}, "Metrics")
resources_ok = _validate(resources, {"condition_id", "stage"}, "Resources")
predictions_ok = _validate(predictions, {"condition_id", "genome_id", "fold_id", "true_label", "predicted_label"}, "Predictions")
_validate(conditions, {"condition_id", "phenotype", "representation", "total_k"}, "Conditions")
_validate(split_manifest, {"genome_id", "fold_id"}, "Split manifest")


In [ ]:
# Normalize shared condition fields while preserving every source row.
CONDITION_COLUMNS = ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "suffix_length", "prefix_source", "positive_class"]
def _condition_fields(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in CONDITION_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    if "total_k" in frame:
        frame["total_k"] = pd.to_numeric(frame["total_k"], errors="coerce")
    if "prefix_length" in frame:
        frame["prefix_length"] = pd.to_numeric(frame["prefix_length"], errors="coerce")
    return frame

metrics = _condition_fields(metrics)
resources = _condition_fields(resources)
predictions = _condition_fields(predictions)
conditions = _condition_fields(conditions)
for frame in [metrics, resources, predictions]:
    frame["model_name"] = frame["model_name"].fillna("logistic_regression")
if not metrics_ok and metrics.empty:
    print("No metrics yet: later cells will show empty-data messages and create no figures.")

## 2. Dataset summary

This summary is derived from saved predictions and split metadata. It reports genomes, classes, class balance, folds, clusters, and high-level organism metadata when such columns were included in a manifest.

In [ ]:
def dataset_summary(prediction_table: pd.DataFrame, split_table: pd.DataFrame) -> pd.DataFrame:
    if prediction_table.empty:
        return pd.DataFrame(columns=["dataset", "phenotype", "genomes", "classes", "class_balance", "folds", "clusters"])
    rows = []
    for keys, group in prediction_table.groupby(["dataset", "phenotype"], dropna=False):
        dataset, phenotype = keys
        labels = group[["genome_id", "true_label"]].drop_duplicates("genome_id")
        counts = labels["true_label"].value_counts(dropna=False).to_dict()
        ids = set(labels["genome_id"].astype(str))
        split = split_table[split_table["genome_id"].astype(str).isin(ids)] if not split_table.empty and "genome_id" in split_table else pd.DataFrame()
        organism_columns = [c for c in ["species", "organism", "genus", "taxon", "organism_name"] if c in split]
        organism_composition = {c: {str(k): int(v) for k, v in split[c].dropna().value_counts().head(20).items()} for c in organism_columns}
        rows.append({
            "dataset": dataset, "phenotype": phenotype, "genomes": len(labels),
            "classes": len(counts), "class_balance": json.dumps({str(k): int(v) for k, v in counts.items()}),
            "folds": split["fold_id"].nunique() if "fold_id" in split else group["fold_id"].nunique(),
            "clusters": split["cluster_id"].nunique() if "cluster_id" in split else pd.NA,
            "organism_composition": json.dumps(organism_composition) if organism_composition else pd.NA,
        })
    return pd.DataFrame(rows)

dataset_table = dataset_summary(predictions, split_manifest)
display(dataset_table)
if dataset_table.empty:
    print("Dataset summary unavailable until prediction or split-manifest outputs exist.")

## 2A. Overview of every completed test run

Each row is one saved experiment directory. RNN settings and the transformer attention-free ablation remain separate instead of being averaged together. `coverage` distinguishes complete five-fold comparisons from partial pilots; runtime and peak RAM come from the matching resource file.

**Dated repository snapshot (2026-09-18, default `results/reviewer_benchmark` only; 28 runs):** 13 logistic-regression conditions, 9 HGBM conditions, 4 partial RNN pilots, one five-fold transformer, and one five-fold attention-free ablation. The transformer reached mean balanced accuracy 0.605 and ROC AUC 0.708 in 79.2 minutes with 1.48 GiB peak RAM; the ablation reached 0.609 and 0.713 in 4.9 minutes with 0.62 GiB. The generated tables below always reflect the currently selected output directory. Model-level averages are an inventory, not a head-to-head ranking, because the models cover different condition sets.

In [ ]:
def _run_directory(frame: pd.DataFrame) -> pd.Series:
    if frame.empty or "_source_file" not in frame:
        return pd.Series(index=frame.index, dtype=object)
    return frame["_source_file"].map(lambda value: str(Path(value).parent))

def _run_variant(run_directory: str) -> str:
    parts = Path(run_directory).parts
    if "models" in parts:
        tail = parts[parts.index("models") + 2:]
        return tail[0] if tail else "default"
    return "default"

def completed_run_overview(metric_table: pd.DataFrame, resource_table: pd.DataFrame) -> pd.DataFrame:
    if metric_table.empty:
        return pd.DataFrame()
    metric_table = metric_table.copy()
    metric_table["run_directory"] = _run_directory(metric_table)
    resource_table = resource_table.copy()
    resource_table["run_directory"] = _run_directory(resource_table)
    rows = []
    for run_directory, group in metric_table.groupby("run_directory", dropna=False):
        first = group.iloc[0]
        model_name, variant = str(first["model_name"]), _run_variant(str(run_directory))
        run_resources = resource_table[resource_table["run_directory"].eq(run_directory)]
        overall = run_resources[run_resources["stage"].eq("overall")] if "stage" in run_resources else run_resources
        runtime = pd.to_numeric(overall.get("wall_time_seconds", pd.Series(dtype=float)), errors="coerce").sum(min_count=1)
        peak_rss = pd.to_numeric(run_resources.get("peak_rss_bytes", pd.Series(dtype=float)), errors="coerce").max()
        folds = sorted(pd.to_numeric(group["fold_id"], errors="coerce").dropna().astype(int).unique().tolist())
        test_kind = "attention-free ablation" if model_name == "transformer" and "layers-0" in variant else model_name
        rows.append({
            "test": test_kind, "variant": variant, "condition_id": first["condition_id"],
            "representation": first["representation"], "total_k": first["total_k"], "prefix": first["prefix"],
            "folds": ",".join(map(str, folds)), "folds_run": len(folds),
            "coverage": "complete (5 folds)" if len(folds) == 5 else f"pilot ({len(folds)} fold{'s' if len(folds) != 1 else ''})",
            "test_samples": int(pd.to_numeric(group.get("n_samples"), errors="coerce").sum()),
            "balanced_accuracy": pd.to_numeric(group["balanced_accuracy"], errors="coerce").mean(),
            "roc_auc": pd.to_numeric(group.get("roc_auc"), errors="coerce").mean(),
            "sensitivity": pd.to_numeric(group.get("sensitivity"), errors="coerce").mean(),
            "specificity": pd.to_numeric(group.get("specificity"), errors="coerce").mean(),
            "epochs_mean": pd.to_numeric(group.get("epochs_run"), errors="coerce").mean(),
            "runtime_minutes": runtime / 60 if pd.notna(runtime) else np.nan,
            "peak_ram_gib": peak_rss / (1024 ** 3) if pd.notna(peak_rss) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["test", "condition_id", "variant"], kind="stable").reset_index(drop=True)

run_overview = completed_run_overview(metrics, resources)
display(run_overview.style.format({c: "{:.3f}" for c in ["balanced_accuracy", "roc_auc", "sensitivity", "specificity", "epochs_mean", "runtime_minutes", "peak_ram_gib"] if c in run_overview}))
if not run_overview.empty:
    model_overview = run_overview.groupby(["test", "coverage"], dropna=False).agg(runs=("condition_id", "size"), mean_balanced_accuracy=("balanced_accuracy", "mean"), mean_roc_auc=("roc_auc", "mean"), total_runtime_minutes=("runtime_minutes", "sum"), max_peak_ram_gib=("peak_ram_gib", "max")).reset_index()
    display(model_overview.style.format({"mean_balanced_accuracy": "{:.3f}", "mean_roc_auc": "{:.3f}", "total_runtime_minutes": "{:.1f}", "max_peak_ram_gib": "{:.2f}"}))
else:
    model_overview = pd.DataFrame()
    print("No completed test runs found.")

## 3. Full versus prefix-filtered performance

Fold-level scores remain the source of truth. The summary below computes condition-level means, medians, standard deviations, and percentile intervals from those tidy rows; no plotted value is entered as a constant. Chance balanced accuracy is `1 / n_classes`.

In [ ]:
def _percentile_interval(values: pd.Series, confidence: float = CONFIDENCE) -> tuple[float, float]:
    values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if not len(values):
        return (np.nan, np.nan)
    alpha = (1 - confidence) / 2
    return (float(np.quantile(values, alpha)), float(np.quantile(values, 1 - alpha)))

def summarize_performance(metric_table: pd.DataFrame) -> pd.DataFrame:
    required = {"condition_id", "fold_id", "balanced_accuracy"}
    if metric_table.empty or not required.issubset(metric_table.columns):
        return pd.DataFrame()
    table = _condition_fields(metric_table)
    rows = []
    grouping = ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source"]
    for keys, group in table.groupby(grouping, dropna=False):
        if fold_confidence_interval is not None:
            seed = BOOTSTRAP_SEED + sum(ord(char) for char in str(keys[0]))
            interval = fold_confidence_interval(group, value_column="balanced_accuracy", seed=seed)
            lo, hi = interval["ci_lower"], interval["ci_upper"]
        else:
            lo, hi = _percentile_interval(group["balanced_accuracy"])
        classes = pd.to_numeric(group.get("n_classes", pd.Series(dtype=float)), errors="coerce").dropna()
        n_classes = int(classes.iloc[0]) if len(classes) else int(group.get("true_label", pd.Series(dtype=object)).nunique()) or 2
        row = dict(zip(grouping, keys))
        row.update({"n_folds": group["fold_id"].nunique(), "balanced_accuracy_mean": group["balanced_accuracy"].mean(), "balanced_accuracy_median": group["balanced_accuracy"].median(), "balanced_accuracy_std": group["balanced_accuracy"].std(ddof=1), "ci_lower": lo, "ci_upper": hi, "n_classes": n_classes, "chance_balanced_accuracy": chance_balanced_accuracy(n_classes)})
        for metric in ["sensitivity", "specificity", "macro_f1", "roc_auc"]:
            values = pd.to_numeric(group.get(metric, pd.Series(dtype=float)), errors="coerce")
            row[f"{metric}_mean"] = values.mean()
            if fold_confidence_interval is not None and values.notna().any():
                metric_interval = fold_confidence_interval(values.dropna().to_numpy(), seed=seed + sum(ord(char) for char in metric))
                row[f"{metric}_ci_lower"] = metric_interval["ci_lower"]
                row[f"{metric}_ci_upper"] = metric_interval["ci_upper"]
        rows.append(row)
    return pd.DataFrame(rows)

performance = summarize_performance(metrics)
display(performance.head(20))
if performance.empty:
    print("Performance summaries will appear after condition metrics are saved.")

def paired_difference_table(metric_table: pd.DataFrame, summary_table: pd.DataFrame) -> pd.DataFrame:
    """Align full and filtered fold scores by fold ID and compute full-minus-filtered intervals."""
    if metric_table.empty or summary_table.empty:
        return pd.DataFrame()
    rows = []
    filtered_conditions = summary_table[summary_table["representation"].eq("prefix_filtered")]
    for condition in filtered_conditions.itertuples(index=False):
        same = metric_table[(metric_table["model_name"] == condition.model_name) & (metric_table["dataset"] == condition.dataset) & (metric_table["phenotype"] == condition.phenotype) & (metric_table["total_k"] == condition.total_k)]
        full = same[same["representation"].eq("full")]
        current = same[same["condition_id"].eq(condition.condition_id)]
        if full.empty or current.empty: continue
        aligned = full[["fold_id", "balanced_accuracy"]].merge(current[["fold_id", "balanced_accuracy"]], on="fold_id", suffixes=("_full", "_filtered"), validate="one_to_one")
        if aligned.empty: continue
        differences = aligned["balanced_accuracy_full"] - aligned["balanced_accuracy_filtered"]
        if paired_difference_confidence_interval is not None:
            full_values = aligned[["fold_id", "balanced_accuracy_full"]].rename(columns={"balanced_accuracy_full": "score"})
            filtered_values = aligned[["fold_id", "balanced_accuracy_filtered"]].rename(columns={"balanced_accuracy_filtered": "score"})
            interval = paired_difference_confidence_interval(full_values, filtered_values, id_column="fold_id", value_column="score", seed=BOOTSTRAP_SEED)
            lo, hi = interval["ci_lower"], interval["ci_upper"]
        else:
            lo, hi = _percentile_interval(differences)
        rows.append({"condition_id": condition.condition_id, "model_name": condition.model_name, "dataset": condition.dataset, "phenotype": condition.phenotype, "total_k": condition.total_k, "prefix": condition.prefix, "prefix_length": condition.prefix_length, "paired_n_folds": len(aligned), "paired_difference_mean": differences.mean(), "paired_difference_median": differences.median(), "paired_ci_lower": lo, "paired_ci_upper": hi})
    return pd.DataFrame(rows)

paired_difference_summary = paired_difference_table(metrics, performance)
display(paired_difference_summary.head(20))
if paired_difference_summary.empty and not performance.empty:
    print("Paired intervals will appear when matching full and filtered fold rows are available.")

## 4. Prefix robustness

Each seeded prefix is an observation. Historical prefixes are overlaid only at their matching length and are assigned a descriptive percentile against the seeded distribution for the same phenotype and `total_k`; they are never used to select a test-set winner.

In [ ]:
def prefix_robustness_table(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty or "representation" not in summary:
        return pd.DataFrame()
    filtered = summary[summary["representation"].eq("prefix_filtered")].copy()
    if filtered.empty:
        return filtered
    filtered["prefix_role"] = np.where(filtered["prefix_source"].astype(str).str.contains("historical", na=False), "historical", "seeded")
    random = filtered[filtered["prefix_role"].eq("seeded")]
    percentiles = []
    for _, row in filtered.iterrows():
        reference = random[(random["model_name"] == row["model_name"]) & (random["phenotype"] == row["phenotype"]) & (random["total_k"] == row["total_k"]) & (random["prefix_length"] == row["prefix_length"])]["balanced_accuracy_mean"]
        value = row["balanced_accuracy_mean"]
        percentiles.append(float((reference <= value).mean() * 100) if len(reference) else np.nan)
    filtered["historical_percentile"] = percentiles
    return filtered

prefix_summary = prefix_robustness_table(performance)
prefix_distribution = (prefix_summary[prefix_summary["prefix_role"].eq("seeded")] if not prefix_summary.empty else pd.DataFrame())
display(prefix_summary.head(30))
if prefix_summary.empty:
    print("Prefix robustness is unavailable until filtered-condition metrics exist.")

## 5. Accuracy versus compression, peak RAM, and runtime

Resource rows are stage measurements. Runtime is summed across stages and peak RSS is the maximum observed stage RSS. Repeated condition metadata are de-duplicated before joining to performance.

In [ ]:
def summarize_resources(resource_table: pd.DataFrame) -> pd.DataFrame:
    if resource_table.empty or "condition_id" not in resource_table:
        return pd.DataFrame()
    table = _condition_fields(resource_table)
    rows = []
    for keys, group in table.groupby(["condition_id", "model_name"], dropna=False):
        condition_id, model_name = keys
        row = {"condition_id": condition_id, "model_name": model_name}
        for column in ["dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source", "input_genomes", "total_bases", "retained_token_count", "retention_fraction", "feature_space_size", "observed_features", "matrix_rows", "matrix_columns", "matrix_nnz", "serialized_size_bytes"]:
            if column in group and group[column].notna().any():
                row[column] = group[column].dropna().iloc[0]
        timed_group = group[group["stage"].eq("overall")] if "stage" in group and group["stage"].eq("overall").any() else group
        durations = pd.to_numeric(timed_group.get("wall_time_seconds", pd.Series(dtype=float)), errors="coerce")
        rss = pd.to_numeric(group.get("peak_rss_bytes", pd.Series(dtype=float)), errors="coerce")
        row["runtime_seconds"] = durations.sum(min_count=1)
        row["peak_rss_bytes"] = rss.max()
        rows.append(row)
    return pd.DataFrame(rows)

resource_summary = summarize_resources(resources)
tradeoff = performance.merge(resource_summary, on=[c for c in ["condition_id", "model_name", "dataset", "phenotype", "representation", "total_k", "prefix", "prefix_length", "prefix_source"] if c in performance and c in resource_summary], how="left") if not performance.empty and not resource_summary.empty else pd.DataFrame()
if not tradeoff.empty and not paired_difference_summary.empty:
    tradeoff = tradeoff.merge(paired_difference_summary, on=["condition_id", "model_name", "dataset", "phenotype", "total_k", "prefix", "prefix_length"], how="left")
if not tradeoff.empty:
    full_resources = tradeoff[tradeoff["representation"].eq("full")][[c for c in ["model_name", "dataset", "phenotype", "total_k", "peak_rss_bytes", "runtime_seconds", "serialized_size_bytes"] if c in tradeoff]].drop_duplicates(["model_name", "dataset", "phenotype", "total_k"]).rename(columns={"peak_rss_bytes": "full_peak_rss_bytes", "runtime_seconds": "full_runtime_seconds", "serialized_size_bytes": "full_serialized_size_bytes"})
    tradeoff = tradeoff.merge(full_resources, on=["model_name", "dataset", "phenotype", "total_k"], how="left")
    for current, baseline, output in [("peak_rss_bytes", "full_peak_rss_bytes", "peak_rss_reduction_fraction"), ("runtime_seconds", "full_runtime_seconds", "runtime_reduction_fraction"), ("serialized_size_bytes", "full_serialized_size_bytes", "matrix_size_reduction_fraction")]:
        if current in tradeoff and baseline in tradeoff:
            tradeoff[output] = np.where(tradeoff["representation"].eq("prefix_filtered"), 1 - pd.to_numeric(tradeoff[current], errors="coerce") / pd.to_numeric(tradeoff[baseline], errors="coerce"), np.nan)
display(tradeoff.head(20))
if tradeoff.empty:
    print("Accuracy/resource trade-offs will appear once both metrics and resource tables are available.")

## 6. Gentamicin sensitivity/specificity and optional ResFinder overlay

Sensitivity and specificity are taken from the runner's explicit positive-class orientation. ResFinder is optional and is shown as an unpaired external reference unless its rows carry the same benchmark genome IDs.

In [ ]:
gentamicin = performance[performance["phenotype"].astype(str).str.lower().str.contains("gentamicin", na=False)].copy() if not performance.empty else pd.DataFrame()
if not gentamicin.empty and not paired_difference_summary.empty:
    gentamicin = gentamicin.merge(paired_difference_summary, on=["condition_id", "model_name", "dataset", "phenotype", "total_k", "prefix", "prefix_length"], how="left")
if not gentamicin.empty and not tradeoff.empty:
    gentamicin = gentamicin.merge(tradeoff[[c for c in ["condition_id", "model_name", "retention_fraction", "peak_rss_reduction_fraction", "runtime_reduction_fraction", "matrix_size_reduction_fraction"] if c in tradeoff]], on=["condition_id", "model_name"], how="left")
gentamicin_display = gentamicin[[c for c in ["condition_id", "model_name", "representation", "total_k", "prefix", "prefix_source", "balanced_accuracy_mean", "sensitivity_mean", "specificity_mean", "ci_lower", "ci_upper", "paired_difference_mean", "paired_ci_lower", "paired_ci_upper", "retention_fraction", "peak_rss_reduction_fraction", "runtime_reduction_fraction", "matrix_size_reduction_fraction"] if c in gentamicin]]
display(gentamicin_display)
if gentamicin.empty:
    print("Gentamicin sensitivity/specificity will appear after gentamicin metrics are saved.")

resfinder_summary = pd.DataFrame()
if prediction_metrics is not None and not resfinder.empty and {"genome_id", "true_label", "predicted_label"}.issubset(resfinder.columns):
    try:
        values = prediction_metrics(resfinder["true_label"], resfinder["predicted_label"], scores=resfinder["score"] if "score" in resfinder else None, positive_label=resfinder["positive_class"].dropna().iloc[0] if "positive_class" in resfinder and resfinder["positive_class"].notna().any() else None)
        resfinder_summary = pd.DataFrame([{"source": "ResFinder (external/unpaired unless IDs match)", **{k: v for k, v in values.items() if k in {"balanced_accuracy", "sensitivity", "specificity", "roc_auc", "n_samples"}}}])
    except Exception as error:
        print(f"ResFinder table found but could not be scored: {error}")
elif not resfinder.empty and prediction_metrics is None:
    print("ResFinder table found, but benchmark metrics dependencies are unavailable; skipping it safely.")
display(resfinder_summary)
if resfinder.empty:
    print("No ResFinder table found; its absence does not block the core benchmark.")

## 7. QC and supplement

The QC table checks condition/fold uniqueness, prediction coverage, resource units, and split leakage where cluster IDs are available. Confusion matrices and raw fold values are retained as exportable supplement tables.

In [ ]:
qc_rows = []
def qc(check: str, passed: bool, detail: str) -> None:
    qc_rows.append({"check": check, "passed": bool(passed), "detail": detail})

if metrics_ok:
    qc("unique model-condition-fold metrics", not metrics.duplicated(["model_name", "condition_id", "fold_id"]).any(), "No duplicate model/condition/fold score rows")
if predictions_ok:
    qc("unique model-condition-fold-genome predictions", not predictions.duplicated(["model_name", "condition_id", "fold_id", "genome_id"]).any(), "No duplicate model/condition out-of-fold predictions")
    qc("prediction labels present", predictions[["true_label", "predicted_label"]].notna().all().all(), "True and predicted labels are non-null")
if resources_ok:
    for column in ["wall_time_seconds", "cpu_time_seconds", "peak_rss_bytes", "serialized_size_bytes"]:
        if column in resources:
            values = pd.to_numeric(resources[column], errors="coerce")
            qc(f"non-negative {column}", bool((values.dropna() >= 0).all()), "Resource values use seconds or bytes as documented")
if not split_manifest.empty and {"genome_id", "fold_id", "cluster_id"}.issubset(split_manifest.columns):
    leakage = split_manifest.groupby("cluster_id")["fold_id"].nunique()
    qc("cluster does not cross folds", bool((leakage <= 1).all()), f"{int((leakage > 1).sum())} leaking clusters")

qc_table = pd.DataFrame(qc_rows)
fold_values = metrics.copy() if not metrics.empty else pd.DataFrame()
confusion_table = metrics[[c for c in ["condition_id", "model_name", "phenotype", "fold_id", "confusion_matrix", "classes"] if c in metrics]] if not metrics.empty else pd.DataFrame()
failed_conditions = failures.copy()
display(qc_table)
display(failed_conditions.head(20))
if qc_table.empty:
    print("QC will populate as saved outputs become available.")

## Figure definitions

The metadata below is exported with each figure. Full points are condition-level means across folds; filtered points/boxes use every seeded prefix condition. Error bars are 95% percentile intervals across folds. Box = Q1–Q3, center line = median, whiskers = most extreme values within 1.5×IQR, and individual points are overlaid. Chance lines are balanced-accuracy references of `1 / number_of_classes`.

In [ ]:
# Match the manuscript figures' compact typography, muted colors, and white grid.
FULL_COLOR = "#4C72B0"
FILTERED_COLOR = "#DD8452"
REFERENCE_LINE = {"color": "0.45", "linestyle": "--", "linewidth": 0.9}
if PLOTTING_AVAILABLE:
    sns.set_theme(context="paper", style="darkgrid", palette="deep", rc={
        "figure.facecolor": "white",
        "axes.facecolor": "#EAEAF2",
        "grid.color": "white",
        "axes.titleweight": "semibold",
        "legend.frameon": True,
        "svg.fonttype": "none",
    })

FIGURE_METADATA = {
    "Figure A": {"title": "Accuracy–resource trade-off", "chance_line": "balanced accuracy = 1 / n_classes", "points": "condition mean across folds; error bars are 95% fold percentile intervals", "boxes": None, "whiskers": None},
    "Figure B": {"title": "Sensitivity to prefix choice", "chance_line": "balanced accuracy = 1 / n_classes", "points": "one point per seeded prefix condition, mean across folds; historical prefix is a distinct point", "boxes": "Q1–Q3 of seeded prefix condition means", "whiskers": "most extreme observations within 1.5×IQR"},
    "Figure C": {"title": "Gentamicin balanced accuracy by model and representation", "chance_line": None, "points": "hollow circles are individual fold outliers; an optional ResFinder tick is a single external score; larger runs show full and historical conditions here, with the seeded prefix distributions in Figure B", "boxes": "Q1–Q3 of the individual cross-validation fold scores", "whiskers": "most extreme scores within 1.5×IQR"},
    "Figure S5": {"title": "Gentamicin paired and resource summaries", "chance_line": "balanced accuracy = 1 / n_classes", "points": "condition means and 95% fold intervals, paired full-minus-filtered intervals, and matched resource reductions; dotted ResFinder line is an external reference when available", "boxes": None, "whiskers": None},
    "Figure S2": {"title": "Fold-level paired performance differences", "chance_line": "zero means equal balanced accuracy", "points": "one full-minus-filtered difference per identical fold", "boxes": "Q1–Q3 across paired folds", "whiskers": "most extreme observations within 1.5×IQR"},
    "Figure S3": {"title": "Condition-level resource measurements", "chance_line": "not applicable", "points": "one condition summary for runtime, peak RSS, matrix size, or retained-token fraction", "boxes": None, "whiskers": None},
    "Figure S4": {"title": "Dataset class and organism composition", "chance_line": "not applicable", "points": "deduplicated genome counts; organism panel is shown when metadata is available", "boxes": None, "whiskers": None},
    "Figure S1": {"title": "Fold-level confusion matrices", "chance_line": None, "points": "one heatmap per saved condition/fold confusion matrix", "boxes": None, "whiskers": None},
}

def _caption(name: str) -> str:
    metadata = FIGURE_METADATA[name]
    reference = f"Chance reference: {metadata['chance_line']}. " if metadata["chance_line"] else ""
    return (f"{name}: {metadata['title']}. {reference}"
            f"Points: {metadata['points']}. "
            + (f"Boxes: {metadata['boxes']}; whiskers: {metadata['whiskers']}." if metadata["boxes"] else ""))

def _finish_figure(name: str, figure: Any) -> Any:
    figure.tight_layout()
    print(_caption(name))
    return figure

## Figures A–C

These cells only plot the tidy summaries above. With no saved results they print a useful message and return without creating an empty publication figure.

In [ ]:
figures: dict[str, Any] = {}
if not PLOTTING_AVAILABLE:
    print("Figures skipped: install matplotlib and seaborn to render them.")
elif tradeoff.empty:
    print("Figure A skipped: no joined metrics/resource rows.")
else:
    phenotypes = [p for p in tradeoff["phenotype"].dropna().unique()]
    fig, axes = plt.subplots(max(1, len(phenotypes)), 1, figsize=(9, 4 * max(1, len(phenotypes))), squeeze=False)
    for axis, phenotype in zip(axes.flat, phenotypes):
        subset = tradeoff[tradeoff["phenotype"].eq(phenotype)].copy()
        subset["x_retention_fraction"] = pd.to_numeric(subset.get("retention_fraction"), errors="coerce")
        for (model_name, representation), points in subset.groupby(["model_name", "representation"], dropna=False):
            color = FULL_COLOR if representation == "full" else FILTERED_COLOR
            marker = "o" if model_name == "logistic_regression" else "D"
            points = points.dropna(subset=["x_retention_fraction", "balanced_accuracy_mean"])
            if points.empty: continue
            axis.errorbar(points["x_retention_fraction"], points["balanced_accuracy_mean"], yerr=[points["balanced_accuracy_mean"] - points["ci_lower"], points["ci_upper"] - points["balanced_accuracy_mean"]], fmt=marker, markersize=4.5, linewidth=1, capsize=2, alpha=0.8, color=color, label=f"{model_name}: {representation}")
        chance = subset["chance_balanced_accuracy"].dropna().iloc[0] if subset["chance_balanced_accuracy"].notna().any() else 0.5
        axis.axhline(chance, label="Chance", **REFERENCE_LINE)
        axis.set(title=str(phenotype).replace("_", " ").capitalize(), xlabel="Retained-token fraction", ylabel="Balanced accuracy")
        axis.legend()
    figures["Figure A"] = _finish_figure("Figure A", fig)
    display(fig)

In [ ]:
if not PLOTTING_AVAILABLE:
    print("Figure B skipped: install matplotlib and seaborn to render it.")
elif prefix_distribution.empty:
    print("Figure B skipped: no seeded prefix conditions.")
else:
    facet_columns = ["model_name", "phenotype", "prefix_length"]
    facets = list(prefix_distribution.groupby(facet_columns, sort=False))
    ncols = min(3, len(facets))
    nrows = (len(facets) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 2.5 * nrows), squeeze=False)
    for axis, ((model, phenotype, prefix_length), subset) in zip(axes.flat, facets):
        historical = prefix_summary[
            prefix_summary["prefix_role"].eq("historical")
            & prefix_summary["model_name"].eq(model)
            & prefix_summary["phenotype"].eq(phenotype)
            & prefix_summary["prefix_length"].eq(prefix_length)
        ]
        k_order = sorted(set(subset["total_k"]) | set(historical["total_k"]))
        sns.boxplot(data=subset, y="total_k", x="balanced_accuracy_mean", order=k_order,
                    orient="h", ax=axis, color=FULL_COLOR, whis=1.5, showfliers=False, width=0.53)
        sns.stripplot(data=subset, y="total_k", x="balanced_accuracy_mean", order=k_order,
                      orient="h", ax=axis, color="0.26", alpha=0.65, jitter=0.13, size=3)
        if not historical.empty:
            axis.scatter(historical["balanced_accuracy_mean"],
                         historical["total_k"].map({k: i for i, k in enumerate(k_order)}),
                         color=FILTERED_COLOR, marker="D", s=23, zorder=4, label="Historical prefix")
        chance = subset["chance_balanced_accuracy"].dropna()
        if not chance.empty:
            axis.axvline(chance.iloc[0], label="Chance", **REFERENCE_LINE)
        name = str(phenotype).replace("_", " ").capitalize()
        model_label = str(model).replace("_", " ")
        axis.set(title=f"{name} · prefix length {prefix_length}\n{model_label}",
                 xlabel="Balanced accuracy", ylabel="Total k-mer length")
        axis.set_facecolor("white")
        axis.grid(visible=False)
        for spine in axis.spines.values():
            spine.set_color("#505050")
            spine.set_linewidth(0.8)
        if axis.get_legend_handles_labels()[0]:
            axis.legend(loc="best", fontsize=7)
    for axis in list(axes.flat)[len(facets):]:
        axis.set_visible(False)
    figures["Figure B"] = _finish_figure("Figure B", fig)
    display(fig)

In [ ]:
if not PLOTTING_AVAILABLE:
    print("Figure C skipped: install matplotlib and seaborn to render it.")
elif gentamicin.empty:
    print("Figure C skipped: no gentamicin metrics.")
else:
    plot_data = gentamicin.sort_values(["total_k", "model_name", "representation", "prefix"])
    if len(plot_data) > 12:
        # The prefix-distribution figure already shows all seeded prefixes.
        core = plot_data["representation"].eq("full") | plot_data["prefix_source"].eq("historical")
        if core.any():
            plot_data = plot_data[core]

    fold_plot = metrics.merge(
        plot_data[["condition_id", "model_name"]].drop_duplicates(),
        on=["condition_id", "model_name"], how="inner",
    ).dropna(subset=["balanced_accuracy"])
    if fold_plot.empty:
        print("Figure C skipped: no gentamicin fold scores.")
    else:
        model_names = {
            "logistic_regression": "Logistic regression",
            "hist_gradient_boosting": "HGB",
            "small_rnn": "RNN",
        }
        rows = []
        labels = []
        for point in plot_data.itertuples():
            scores = fold_plot.loc[
                fold_plot["condition_id"].eq(point.condition_id)
                & fold_plot["model_name"].eq(point.model_name), "balanced_accuracy"
            ]
            if scores.empty:
                continue
            model = model_names.get(point.model_name, str(point.model_name).replace("_", " "))
            representation = (
                f"full {int(point.total_k)}-mer" if point.representation == "full"
                else f"{point.prefix}, {int(point.total_k) - int(point.prefix_length)}"
            )
            labels.append(f"{model}, {representation}")
            rows.append(scores.to_numpy())
        external = resfinder_summary.get("balanced_accuracy", pd.Series(dtype=float)).dropna()
        offset = int(not external.empty)
        positions = np.arange(offset, len(rows) + offset)
        fig, axis = plt.subplots(figsize=(5.6, max(2.15, 0.235 * (len(rows) + offset) + 0.70)))
        line = {"color": "#666666", "linewidth": 0.85}
        axis.boxplot(
            rows, orientation="horizontal", positions=positions, widths=0.76,
            whis=1.5, manage_ticks=False, patch_artist=True,
            boxprops={"facecolor": "#5875A4", "edgecolor": "#4B4B4B", "linewidth": 0.85},
            medianprops=line, whiskerprops=line, capprops=line,
            flierprops={"marker": "o", "markerfacecolor": "none",
                        "markeredgecolor": "#707070", "markersize": 3},
        )
        if offset:
            axis.vlines(external.iloc[0], -0.37, 0.37, color="#666666", linewidth=0.9)
            labels.insert(0, "ResFinder")
        axis.set_yticks(np.arange(len(labels)), labels=labels)
        axis.set_ylim(len(labels) - 0.5, -0.5)
        axis.set_xlabel("Balanced Accuracy")
        axis.tick_params(axis="both", length=0)
        axis.grid(axis="y", visible=False)
        minimum = min(float(row.min()) for row in rows)
        if offset:
            minimum = min(minimum, float(external.iloc[0]))
        axis.set_xlim(min(0.47, max(0, minimum - 0.03)), 1.01)
        axis.set_xticks(np.arange(0.5, 1.01, 0.1) if minimum >= 0.47 else axis.get_xticks()[
            (axis.get_xticks() >= axis.get_xlim()[0]) & (axis.get_xticks() <= 1.01)
        ])
        figures["Figure C"] = _finish_figure("Figure C", fig)
        display(fig)

In [ ]:
# Essential QC supplement: saved confusion matrices, one heatmap per condition/fold.
if not PLOTTING_AVAILABLE:
    print("Figure S1 skipped: install matplotlib and seaborn to render it.")
elif confusion_table.empty:
    print("Figure S1 skipped: no saved confusion matrices.")
else:
    matrices = confusion_table.head(12)
    ncols = min(4, len(matrices))
    nrows = (len(matrices) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.45 * ncols, 2.15 * nrows), squeeze=False)
    for axis, (_, row) in zip(axes.flat, matrices.iterrows()):
        values = np.asarray(json.loads(row["confusion_matrix"]))
        sns.heatmap(values, annot=True, fmt="g", cbar=False, ax=axis, cmap="Blues")
        condition = str(row.get("condition_id", "")).split("__", 2)[-1].replace("__", " · ")
        model = str(row.get("model_name", "")).replace("_", " ")
        axis.set(title=f"{condition} · fold {row.get('fold_id', '')}\n{model}", xlabel="Predicted", ylabel="True")
    for axis in list(axes.flat)[len(matrices):]:
        axis.set_visible(False)
    figures["Figure S1"] = _finish_figure("Figure S1", fig)
    display(fig)

In [ ]:
# Supplementary paired-fold, resource, and dataset-composition figures.
paired_fold_rows = []
if not metrics.empty and {"condition_id", "fold_id", "balanced_accuracy"}.issubset(metrics.columns):
    for keys, filtered in metrics[metrics["representation"].eq("prefix_filtered")].groupby(["model_name", "condition_id"]):
        model_name, condition_id = keys
        first = filtered.iloc[0]
        full = metrics[(metrics["model_name"].eq(model_name)) & (metrics["representation"].eq("full")) & (metrics["dataset"].eq(first["dataset"])) & (metrics["phenotype"].eq(first["phenotype"])) & (metrics["total_k"].eq(first["total_k"]))]
        aligned = full[["fold_id", "balanced_accuracy"]].merge(filtered[["fold_id", "balanced_accuracy"]], on="fold_id", suffixes=("_full", "_filtered"))
        for _, row in aligned.iterrows():
            paired_fold_rows.append({"condition_id": condition_id, "model_name": model_name, "phenotype": first["phenotype"], "prefix_length": first["prefix_length"], "fold_id": row["fold_id"], "difference": row["balanced_accuracy_full"] - row["balanced_accuracy_filtered"]})
paired_fold_values = pd.DataFrame(paired_fold_rows)
if not PLOTTING_AVAILABLE or paired_fold_values.empty:
    print("Figure S2 skipped: no paired fold values.")
else:
    fig, axis = plt.subplots(figsize=(10, max(4, 0.7 * paired_fold_values["phenotype"].nunique())))
    sns.boxplot(data=paired_fold_values, x="difference", y="phenotype", whis=1.5, showfliers=False, color=FULL_COLOR, ax=axis)
    sns.stripplot(data=paired_fold_values, x="difference", y="phenotype", hue="prefix_length", alpha=0.45, ax=axis)
    axis.axvline(0, **REFERENCE_LINE)
    axis.set(xlabel="Full − filtered balanced accuracy", ylabel="Phenotype")
    figures["Figure S2"] = _finish_figure("Figure S2", fig)
    display(fig)

resource_columns = [c for c in ["runtime_seconds", "peak_rss_bytes", "serialized_size_bytes", "retention_fraction"] if c in resource_summary]
if not PLOTTING_AVAILABLE or resource_summary.empty or not resource_columns:
    print("Figure S3 skipped: no condition resource summaries.")
else:
    resource_plot = resource_summary.melt(id_vars=[c for c in ["condition_id", "model_name", "phenotype", "representation"] if c in resource_summary], value_vars=resource_columns, var_name="resource", value_name="value").dropna()
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), squeeze=False)
    for axis, resource in zip(axes.flat, resource_columns):
        subset = resource_plot[resource_plot["resource"].eq(resource)]
        sns.stripplot(data=subset, x="representation", y="value", hue="model_name", alpha=0.6, ax=axis)
        axis.set(title=resource.replace("_", " "), xlabel="Representation", ylabel=resource)
    for axis in axes.flat[len(resource_columns):]: axis.set_visible(False)
    figures["Figure S3"] = _finish_figure("Figure S3", fig)
    display(fig)

composition = predictions[[c for c in ["dataset", "phenotype", "genome_id", "true_label"] if c in predictions]].drop_duplicates() if not predictions.empty else pd.DataFrame()
organism_columns = [c for c in ["species", "organism", "genus", "organism_name"] if c in split_manifest]
if not PLOTTING_AVAILABLE or composition.empty or "true_label" not in composition:
    print("Figure S4 skipped: no prediction-level class composition.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    class_counts = composition.groupby(["phenotype", "true_label"], dropna=False).size().rename("genomes").reset_index()
    sns.barplot(data=class_counts, x="phenotype", y="genomes", hue="true_label", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].set(title="Class composition", xlabel="Phenotype", ylabel="Genomes")
    if organism_columns:
        organism = organism_columns[0]
        organism_counts = split_manifest[organism].dropna().astype(str).value_counts().head(15).rename_axis(organism).rename("genomes").reset_index()
        sns.barplot(data=organism_counts, x="genomes", y=organism, color=FULL_COLOR, ax=axes[1])
        axes[1].set(title=f"Top {organism} groups", xlabel="Genomes", ylabel=organism)
    else:
        axes[1].text(0.5, 0.5, "Organism metadata unavailable", ha="center", va="center")
        axes[1].set_axis_off()
    figures["Figure S4"] = _finish_figure("Figure S4", fig)
    display(fig)

In [ ]:
if not PLOTTING_AVAILABLE:
    print("Figure S5 skipped: install matplotlib and seaborn to render it.")
elif gentamicin.empty:
    print("Figure S5 skipped: no gentamicin metrics.")
else:
    plot_data = gentamicin.sort_values(["total_k", "model_name", "representation", "prefix"]).copy()
    model_labels = {"logistic_regression": "Logistic regression", "hist_gradient_boosting": "HGB"}
    show_model = plot_data["model_name"].nunique() > 1
    show_prefix = len(plot_data) <= 12
    labels = []
    for point in plot_data.itertuples():
        representation = "Full" if point.representation == "full" else "Prefix-filtered"
        if show_prefix and point.representation != "full":
            representation = f"Prefix {point.prefix}"
        model = model_labels.get(point.model_name, str(point.model_name).replace("_", " "))
        labels.append(f"k={point.total_k} · {representation}" + (f" · {model}" if show_model else ""))
    plot_data["plot_label"] = labels
    label_order = list(dict.fromkeys(labels))
    positions = {label: index for index, label in enumerate(label_order)}
    plot_data["plot_y"] = plot_data["plot_label"].map(positions).astype(float)
    for _, group in plot_data.groupby("plot_label", sort=False):
        if len(group) > 1:
            plot_data.loc[group.index, "plot_y"] += np.linspace(-0.31, 0.31, len(group))

    fig, axes = plt.subplots(1, 4, sharey=True, figsize=(10.4, max(2.6, 0.31 * len(label_order) + 1.1)))
    panels = [
        ("balanced_accuracy_mean", "Balanced accuracy", "ci_lower", "ci_upper"),
        ("sensitivity_mean", "Sensitivity", "sensitivity_ci_lower", "sensitivity_ci_upper"),
        ("paired_difference_mean", "Full − filtered BA", "paired_ci_lower", "paired_ci_upper"),
    ]
    for axis, (metric, title, low, high) in zip(axes, panels):
        axis.set(title=title)
        if metric not in plot_data or plot_data[metric].isna().all():
            axis.text(0.5, 0.5, "No matched results", ha="center", va="center", transform=axis.transAxes)
            continue
        for representation, points in plot_data.dropna(subset=[metric]).groupby("representation"):
            color = FULL_COLOR if representation == "full" else FILTERED_COLOR
            intervals = points.dropna(subset=[low, high])
            if not intervals.empty:
                axis.hlines(intervals["plot_y"], intervals[low], intervals[high], color=color, linewidth=1.1)
                for bound in (low, high):
                    axis.scatter(intervals[bound], intervals["plot_y"], color=color, marker="|", s=20, linewidths=1)
            axis.scatter(points[metric], points["plot_y"], color=color, s=22, zorder=3)
        if metric == "balanced_accuracy_mean":
            chance = plot_data["chance_balanced_accuracy"].dropna()
            if not chance.empty:
                axis.axvline(chance.iloc[0], **REFERENCE_LINE)
        if metric == "paired_difference_mean":
            axis.axvline(0, **REFERENCE_LINE)
        if metric in {"balanced_accuracy_mean", "sensitivity_mean"} and not resfinder_summary.empty:
            external = resfinder_summary.get(metric.removesuffix("_mean"), pd.Series(dtype=float)).dropna()
            if not external.empty:
                axis.axvline(external.iloc[0], color="0.30", linestyle=":", label="ResFinder (external)")
                axis.legend(loc="best", fontsize=7)

    resource_axis = axes[3]
    resources_to_plot = {
        "peak_rss_reduction_fraction": ("Peak RAM", FULL_COLOR, "s"),
        "runtime_reduction_fraction": ("Runtime", FILTERED_COLOR, "o"),
        "matrix_size_reduction_fraction": ("Matrix size", "#55A868", "^"),
    }
    for index, (column, (label, color, marker)) in enumerate(resources_to_plot.items()):
        if column in plot_data and plot_data[column].notna().any():
            resource_axis.scatter(plot_data[column], plot_data["plot_y"] + 0.13 * (index - 1),
                                  color=color, marker=marker, s=20, label=label)
    resource_axis.axvline(0, **REFERENCE_LINE)
    if resource_axis.get_legend_handles_labels()[0]:
        resource_axis.legend(loc="best", fontsize=7)
    else:
        resource_axis.text(0.5, 0.5, "No matched results", ha="center", va="center", transform=resource_axis.transAxes)
    resource_axis.set(title="Resource reduction", xlabel="Fraction reduced")
    axes[0].set_yticks(range(len(label_order)), labels=label_order)
    axes[0].set_ylim(len(label_order) - 0.5, -0.5)
    for axis in axes:
        axis.grid(axis="y", visible=False)
    figures["Figure S5"] = _finish_figure("Figure S5", fig)
    display(fig)

## Sequence similarity from saved genome sketches

Each sketch archive produces a heatmap and a distribution of pairwise Sourmash Jaccard estimates. The heatmap groups similar sketches with average linkage for display. The histogram counts each distinct genome pair once and excludes self-comparisons.

By default the notebook looks in the `sketches/` directories beside the benchmark output and any discovered split manifests. Set `REVIEWER_BENCHMARK_SKETCHES` to a `.sig.gz` file or a directory of signature archives to select sketches explicitly. Archives are plotted individually. Both panels use every nonempty sketch.

In [ ]:
def sketch_similarity_figures():
    from scipy.cluster.hierarchy import leaves_list, linkage
    from scipy.spatial.distance import squareform
    from sourmash import load_file_as_signatures

    sketch_override = os.environ.get("REVIEWER_BENCHMARK_SKETCHES")
    if sketch_override:
        requested = _repository_relative_path(sketch_override)
        archives = [requested] if requested.is_file() else sorted(requested.glob("*.sig.gz"))
    else:
        directories = {OUTPUT_ROOT / "sketches"}
        if SPLIT_MANIFEST_PATH:
            directories.add(SPLIT_MANIFEST_PATH.parent / "sketches")
        for source in split_manifest.get("_source_file", pd.Series(dtype=str)).dropna().unique():
            directories.add(Path(source).parent / "sketches")
        archives = sorted({path.resolve() for directory in directories for path in directory.glob("*.sig.gz")})

    if not archives:
        print("Sequence similarity plots skipped: no saved sketches found. Set REVIEWER_BENCHMARK_SKETCHES to select an archive or directory.")
        return

    for index, archive in enumerate(archives, start=1):
        try:
            signatures = sorted(load_file_as_signatures(str(archive)), key=lambda signature: signature.name)
            signatures = [signature for signature in signatures if len(signature.minhash)]
            genome_count = len(signatures)
            if genome_count < 2:
                print(f"Skipping {archive}: at least two nonempty sketches are needed.")
                continue
            sketches = [signature.minhash for signature in signatures]
            count = len(sketches)
            rows, columns = np.triu_indices(count, k=1)
            pairwise = np.array([sketches[i].jaccard(sketches[j]) for i, j in zip(rows, columns)])
        except (OSError, ValueError, TypeError) as error:
            print(f"Skipping {archive}: {error}")
            continue

        order = leaves_list(linkage(1.0 - pairwise, method="average"))
        similarities = squareform(pairwise)
        np.fill_diagonal(similarities, 1.0)
        figure, (heatmap, histogram) = plt.subplots(1, 2, figsize=(10, 4))
        image = heatmap.imshow(similarities[np.ix_(order, order)], cmap="viridis", vmin=0, vmax=1)
        heatmap.set(title="Pairwise similarity", xlabel="Genomes (clustered)", ylabel="Genomes (clustered)")
        heatmap.set_xticks([])
        heatmap.set_yticks([])
        figure.colorbar(image, ax=heatmap, label="Sketch Jaccard similarity")
        histogram.hist(pairwise, bins=np.linspace(0, 1, 31), color=FULL_COLOR)
        histogram.set(title="Distribution across genome pairs", xlabel="Sketch Jaccard similarity",
                      ylabel="Genome pairs", xlim=(0, 1))
        figure.suptitle(f"{archive.name[:16]} · k={sketches[0].ksize} · {count:,} nonempty sketches")
        name = f"Sketch similarity {index}"
        FIGURE_METADATA[name] = {
            "title": f"Genome sketch similarity ({archive.name})",
            "chance_line": None,
            "points": f"{count:,} nonempty sketches; {len(pairwise):,} distinct pairs in the histogram",
            "boxes": None,
            "whiskers": None,
        }
        print(f"Sketch archive: {archive}")
        figures[name] = _finish_figure(name, figure)
        plt.show()


if not PLOTTING_AVAILABLE:
    print("Sequence similarity plots skipped: install matplotlib and seaborn.")
else:
    try:
        sketch_similarity_figures()
    except ImportError as error:
        print(f"Sequence similarity plots skipped: {error}")

## 8. Export publication-ready figures and summary tables

Exports are opt-in via `REVIEWER_BENCHMARK_EXPORT=1`; this keeps the notebook read-only by default. The exported tables are the exact tidy inputs used for figures and QC, together with figure-caption metadata.

In [ ]:
export_tables = {
    "dataset_summary.csv": dataset_table,
    "run_overview.csv": run_overview,
    "model_overview.csv": model_overview,
    "performance_summary.csv": performance,
    "prefix_summary.csv": prefix_summary,
    "resource_summary.csv": resource_summary,
    "tradeoff_summary.csv": tradeoff,
    "gentamicin_summary.csv": gentamicin_display,
    "fold_metrics.csv": fold_values,
    "confusion_matrices.csv": confusion_table,
    "qc.csv": qc_table,
    "failed_conditions.csv": failed_conditions,
}

if EXPORT_OUTPUTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for filename, table in export_tables.items():
        table.to_csv(EXPORT_DIR / filename, index=False)
    (EXPORT_DIR / "figure_metadata.json").write_text(json.dumps({name: {**metadata, "caption": _caption(name)} for name, metadata in FIGURE_METADATA.items()}, indent=2) + "\n")
    for name, figure in figures.items():
        stem = name.lower().replace(" ", "_")
        figure.savefig(EXPORT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        figure.savefig(EXPORT_DIR / f"{stem}.svg", bbox_inches="tight")
    print(f"Exported {len(export_tables)} tables and {len(figures)} figures to {EXPORT_DIR.resolve()}")
else:
    print("Exports disabled. Set REVIEWER_BENCHMARK_EXPORT=1 to write CSV, PNG, SVG, and caption metadata files.")